# Minimal Thesis Portfolio Results

This notebook runs one comparable experiment for the Top S&P 500 universe over 2018-2022. It builds five fixed-weight portfolios on the same construction window, evaluates them on the same out-of-sample window, and saves only the final tables and presentation figures.

In [42]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

pio.renderers.default = "plotly_mimetype"

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "portafolios").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "portafolios").exists():
    raise RuntimeError("Could not locate the repository root containing the portafolios package.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from portafolios import (
    Universe,
    StandardizedData,
    EqualWeightConstructor,
    Markowitz,
    NaiveRiskParity,
    HRPStyle,
    HRPRecursive,
    Backtester,
    MonteCarloEngine,
    local_loader,
)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "thesis_minimal_results"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def render_plotly(fig, output_path: Path):
    fig.write_html(output_path, include_plotlyjs=True, full_html=True)
    display(fig)

## Experiment Setup

In [43]:
# Use DATA_FORMAT="yfinance" for a standard yfinance-style MultiIndex CSV.
# Use DATA_FORMAT="processed_ohlcv" for data/processed/data_clean_stock_data.csv.
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "data_clean_stock_data.csv"
DATA_FORMAT = "processed_ohlcv"

FULL_START = "2017-01-01"
FULL_END = "2025-12-31"
CONSTRUCTION_START = "2017-01-01"
CONSTRUCTION_END = "2019-12-31"
BACKTEST_START = "2021-01-01"
BACKTEST_END = "2025-12-31"

TOP_N = 50
ANN_FACTOR = 252
MC_SIMULATIONS = 2000
MC_HORIZON = None  # None means match the out-of-sample backtest length.
MC_INITIAL_VALUE = 1.0
MC_SEED = 42

## Data Loading

In [44]:
def clean_ticker(value):
    ticker = str(value).strip()
    return ticker.lstrip("0") or ticker


def load_processed_ohlcv_close_prices(path: Path, close_field: str = "Close") -> pd.DataFrame:
    raw = pd.read_csv(path, header=None)
    tickers = raw.iloc[0, 1:].ffill()
    fields = raw.iloc[1, 1:]
    data = raw.iloc[3:].copy()
    dates = pd.to_datetime(data.iloc[:, 0])

    prices = {}
    for column_position, (ticker, field) in enumerate(zip(tickers, fields), start=1):
        if str(field).strip().lower() == close_field.lower():
            prices[clean_ticker(ticker)] = pd.to_numeric(data.iloc[:, column_position], errors="coerce").to_numpy()

    return pd.DataFrame(prices, index=dates).sort_index().dropna(axis=1, how="all")


def load_experiment_prices(path: Path, data_format: str) -> pd.DataFrame:
    if data_format == "yfinance":
        return local_loader(path=path, start=FULL_START, end=FULL_END, prefer_adj_close=True)
    if data_format == "processed_ohlcv":
        prices = load_processed_ohlcv_close_prices(path)
        return prices.loc[pd.Timestamp(FULL_START):pd.Timestamp(FULL_END)]
    raise ValueError(f"Unknown DATA_FORMAT: {data_format}")


prices_raw = load_experiment_prices(DATA_PATH, DATA_FORMAT)
prices = prices_raw.dropna(axis=1, how="any").dropna(axis=0, how="any")

if prices.shape[1] < 45:
    raise ValueError(f"Expected a broad Top 50 universe, but only {prices.shape[1]} complete assets are available.")

tickers = list(prices.columns[:TOP_N])
prices = prices.loc[:, tickers]
returns = prices.pct_change().dropna(axis=0, how="any")

market_data = StandardizedData(
    prices=prices,
    returns=returns,
    tickers=tickers,
    metadata={
        "source_path": str(DATA_PATH),
        "data_format": DATA_FORMAT,
        "full_start": FULL_START,
        "full_end": FULL_END,
        "n_assets": len(tickers),
    },
)

universe = Universe(
    loader=market_data,
    tickers=tickers,
    start=FULL_START,
    end=FULL_END,
    construction_start=CONSTRUCTION_START,
    construction_end=CONSTRUCTION_END,
    universe_name="thesis_minimal_results",
    base_output_dir=OUTPUT_DIR / "framework_runs",
    auto_save_data=False,
).prepare_data()

## Portfolio Construction

In [45]:
constructors = [
    ("equal_weight", EqualWeightConstructor(), {}),
    ("markowitz", Markowitz(), {"ret_kind": "simple", "allow_short": False}),
    ("naive_risk_parity", NaiveRiskParity(), {}),
    (
        "hrp_style_deprado_nrp",
        HRPStyle(
            distance="deprado",
            inner=NaiveRiskParity(),
            outer=NaiveRiskParity(),
            n_clusters=5,
            display_name="HRP Style De Prado + NRP",
        ),
        {},
    ),
    (
        "hrp_style_corr_ew_nrp",
        HRPStyle(
            distance="corr",
            inner=EqualWeightConstructor(),
            outer=NaiveRiskParity(),
            n_clusters=5,
            display_name="HRP Style Corr + EW/NRP",
        ),
        {},
    ),
    ("hrp_recursive", HRPRecursive(distance="corr"), {}),
]

for label, constructor, kwargs in constructors:
    universe.build(constructor, label=label, set_active=False, **kwargs)

method_names = [label for label, _, _ in constructors]

## Portfolio Composition

In [46]:
weights = pd.concat(
    {name: universe.get_construction(name).weights for name in method_names},
    axis=1,
).fillna(0.0)

concentration = pd.DataFrame(
    {
        name: {
            "n_assets": int((weights[name].abs() > 1e-12).sum()),
            "max_weight": float(weights[name].max()),
            "top_5_weight": float(weights[name].nlargest(min(5, len(weights))).sum()),
            "herfindahl": float((weights[name] ** 2).sum()),
            "effective_n": float(1.0 / (weights[name] ** 2).sum()),
        }
        for name in method_names
    }
).T

weights.to_csv(TABLE_DIR / "weights.csv")
concentration.to_csv(TABLE_DIR / "concentration.csv")

weights

,equal_weight,markowitz,naive_risk_parity,hrp_style_deprado_nrp,hrp_style_corr_ew_nrp,hrp_recursive
AAPL,0.020408,8.520334e-02,0.017443,0.006862,0.006145,0.000361
ABBV,0.020408,0.000000e+00,0.016450,0.020293,0.006145,0.030409
ABT,0.020408,1.471891e-16,0.023186,0.009122,0.006145,0.001400
ACN,0.020408,1.414686e-17,0.023924,0.009412,0.006145,0.000254
ADBE,0.020408,0.000000e+00,0.017417,0.006852,0.006145,0.000044
AMD,0.020408,6.343832e-02,0.007846,0.003087,0.006145,0.001938
AMGN,0.020408,0.000000e+00,0.020939,0.025831,0.006145,0.033412
AMZN,0.020408,4.207917e-17,0.016255,0.006395,0.006145,0.000039
AVGO,0.020408,1.405816e-17,0.013527,0.005322,0.006145,0.002401
BAC,0.020408,0.000000e+00,0.018645,0.007335,0.006145,0.000191


In [47]:
concentration

,n_assets,max_weight,top_5_weight,herfindahl,effective_n
equal_weight,49.0,0.020408,0.102041,0.020408,49.000000
markowitz,10.0,0.444760,0.795681,0.240782,4.153138
naive_risk_parity,49.0,0.031523,0.142380,0.021600,46.296429
hrp_style_deprado_nrp,49.0,0.110073,0.348742,0.041988,23.816262
hrp_style_corr_ew_nrp,49.0,0.214457,0.560024,0.094410,10.592131
hrp_recursive,49.0,0.147525,0.509387,0.071370,14.011521


## Backtest Results

In [48]:
backtest_results = Backtester.run_all(
    universe,
    start_date=BACKTEST_START,
    end_date=BACKTEST_END,
    ann_factor=ANN_FACTOR,
    attach=True,
)

cumulative_returns = pd.concat(
    {name: backtest_results[name].cumulative_returns for name in method_names},
    axis=1,
).sort_index()

backtest_summary = pd.DataFrame(
    {name: backtest_results[name].summary_metrics for name in method_names}
).T.rename(columns={"annualized_volatility": "volatility"})

backtest_summary = backtest_summary[
    ["total_return", "annualized_return", "volatility", "sharpe_ratio", "max_drawdown"]
]

cumulative_returns.to_csv(TABLE_DIR / "cumulative_returns.csv")
backtest_summary.to_csv(TABLE_DIR / "backtest_summary.csv")

backtest_summary

,total_return,annualized_return,volatility,sharpe_ratio,max_drawdown
equal_weight,0.746847,0.150124,0.128374,1.169427,-0.232583
markowitz,0.642866,0.132560,0.150012,0.883664,-0.177980
naive_risk_parity,0.664295,0.136247,0.118340,1.151317,-0.200679
hrp_style_deprado_nrp,0.681700,0.139215,0.123324,1.128852,-0.223481
hrp_style_corr_ew_nrp,0.582286,0.121941,0.111367,1.094950,-0.170892
hrp_recursive,0.500791,0.107163,0.102034,1.050272,-0.148518


In [49]:
fig_cumulative = go.Figure()
for method in cumulative_returns.columns:
    fig_cumulative.add_trace(
        go.Scatter(
            x=cumulative_returns.index,
            y=cumulative_returns[method],
            mode="lines",
            name=method,
            line=dict(width=2),
        )
    )

fig_cumulative.update_layout(
    title="Cumulative Returns",
    xaxis_title="Date",
    yaxis_title="Cumulative return",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

render_plotly(fig_cumulative, FIGURE_DIR / "cumulative_returns.html")

## Risk Diagnostics

In [50]:
drawdown_series = pd.concat(
    {name: backtest_results[name].drawdown_series for name in method_names},
    axis=1,
).sort_index()

drawdown_series.to_csv(TABLE_DIR / "drawdown_series.csv")

drawdown_series

,equal_weight,markowitz,naive_risk_parity,hrp_style_deprado_nrp,hrp_style_corr_ew_nrp,hrp_recursive
0,,,,,,
2021-01-04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2021-01-05,-0.007502,-0.020742,-0.006915,-0.009366,-0.010058,-0.009738
2021-01-06,0.000000,0.000000,0.000000,0.000000,-0.001020,0.000000
2021-01-07,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2021-01-08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...
2024-12-24,-0.021590,-0.087165,-0.027409,-0.020502,-0.022281,-0.036708
2024-12-26,-0.018580,-0.085237,-0.024464,-0.018203,-0.020000,-0.034412
2024-12-27,-0.021868,-0.091016,-0.026829,-0.022349,-0.022699,-0.037017


In [51]:
fig_drawdown = go.Figure()
for method in drawdown_series.columns:
    fig_drawdown.add_trace(
        go.Scatter(
            x=drawdown_series.index,
            y=drawdown_series[method],
            mode="lines",
            name=method,
            line=dict(width=2),
        )
    )

fig_drawdown.update_layout(
    title="Drawdown Series",
    xaxis_title="Date",
    yaxis_title="Drawdown",
    hovermode="x unified",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

render_plotly(fig_drawdown, FIGURE_DIR / "drawdown_series.html")

## Monte Carlo Simulation

In [52]:
backtest_window_returns = universe.get_returns_window(BACKTEST_START, BACKTEST_END)
mc_horizon = MC_HORIZON or len(backtest_window_returns)

mc_results = MonteCarloEngine.run_all(
    universe,
    horizon=mc_horizon,
    n_simulations=MC_SIMULATIONS,
    initial_value=MC_INITIAL_VALUE,
    seed=MC_SEED,
    attach=True,
)

mc_terminal_values = pd.DataFrame(
    {name: mc_results[name].terminal_values for name in method_names}
)

mc_summary = pd.DataFrame(
    {name: mc_results[name].summary_metrics for name in method_names}
).T[
    ["mean_terminal_value", "median_terminal_value", "std_terminal_value", "prob_loss"]
]

mc_terminal_values.to_csv(TABLE_DIR / "mc_terminal_values.csv", index=False)
mc_summary.to_csv(TABLE_DIR / "mc_summary.csv")

mc_summary

,mean_terminal_value,median_terminal_value,std_terminal_value,prob_loss
equal_weight,1.918933,1.851815,0.503802,0.0095
markowitz,3.274599,3.197987,0.723826,0.0000
naive_risk_parity,1.848246,1.790384,0.438624,0.0075
hrp_style_deprado_nrp,1.786821,1.746220,0.408006,0.0105
hrp_style_corr_ew_nrp,1.654429,1.628191,0.351260,0.0100
hrp_recursive,1.726186,1.696186,0.338996,0.0050


In [53]:
fig_mc = go.Figure()
for method in mc_terminal_values.columns:
    fig_mc.add_trace(
        go.Histogram(
            x=mc_terminal_values[method],
            name=method,
            histnorm="probability density",
            nbinsx=50,
            opacity=0.45,
        )
    )

fig_mc.add_vline(
    x=MC_INITIAL_VALUE,
    line_dash="dash",
    line_color="black",
    annotation_text="Initial value",
    annotation_position="top right",
)

fig_mc.update_layout(
    title="Monte Carlo Terminal Values",
    xaxis_title="Terminal value",
    yaxis_title="Density",
    barmode="overlay",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

render_plotly(fig_mc, FIGURE_DIR / "mc_terminal_histogram.html")

## Summary

In [54]:
best_return = backtest_summary["annualized_return"].idxmax()
best_sharpe = backtest_summary["sharpe_ratio"].idxmax()
lowest_vol = backtest_summary["volatility"].idxmin()
mildest_drawdown = backtest_summary["max_drawdown"].idxmax()
most_concentrated = concentration["herfindahl"].idxmax()
least_concentrated = concentration["herfindahl"].idxmin()
lowest_prob_loss = mc_summary["prob_loss"].idxmin()

summary_lines = [
    f"Highest annualized return: {best_return}.",
    f"Best Sharpe ratio: {best_sharpe}.",
    f"Most stable by volatility: {lowest_vol}.",
    f"Most stable by max drawdown: {mildest_drawdown}.",
    f"Most concentrated by Herfindahl index: {most_concentrated}.",
    f"Most diversified by Herfindahl index: {least_concentrated}.",
    f"Lowest Monte Carlo probability of loss: {lowest_prob_loss}.",
    "Main trade-off: return-seeking methods may improve upside but can become more concentrated and estimation-sensitive; diversified and risk-based methods are usually easier to defend as stable fixed-weight allocations.",
]

summary_text = "\n".join(f"- {line}" for line in summary_lines)
(OUTPUT_DIR / "summary.md").write_text(summary_text, encoding="utf-8")

experiment_config = {
    "data_path": str(DATA_PATH),
    "data_format": DATA_FORMAT,
    "full_start": FULL_START,
    "full_end": FULL_END,
    "construction_start": CONSTRUCTION_START,
    "construction_end": CONSTRUCTION_END,
    "backtest_start": BACKTEST_START,
    "backtest_end": BACKTEST_END,
    "methods": method_names,
    "n_assets": len(tickers),
    "tickers": tickers,
    "ann_factor": ANN_FACTOR,
    "mc_horizon": mc_horizon,
    "mc_simulations": MC_SIMULATIONS,
    "mc_seed": MC_SEED,
    "mc_estimation_windows": {
        name: {
            "estimation_start": str(mc_results[name].estimation_start),
            "estimation_end": str(mc_results[name].estimation_end),
        }
        for name in method_names
    },
}

(OUTPUT_DIR / "experiment_config.json").write_text(
    json.dumps(experiment_config, indent=2),
    encoding="utf-8",
)

print(summary_text)

- Highest annualized return: equal_weight.
- Best Sharpe ratio: equal_weight.
- Most stable by volatility: hrp_recursive.
- Most stable by max drawdown: hrp_recursive.
- Most concentrated by Herfindahl index: markowitz.
- Most diversified by Herfindahl index: equal_weight.
- Lowest Monte Carlo probability of loss: markowitz.
- Main trade-off: return-seeking methods may improve upside but can become more concentrated and estimation-sensitive; diversified and risk-based methods are usually easier to defend as stable fixed-weight allocations.
